# Marketing Mix Model
- Justin Wall, Senior Data Scientist, Arclight Cycling
## Questions to answer
1. Which channels drive revenue? Channel decomposition + ROAS posteriors
2. Are we overspending anywhere? Saturation curves per channel
3. Where should we shift $500K? Budget reallocation optimizer
4. How long does advertising stick? Adstock decay estimates
5. How confident should we be? Posterior credible intervals

### Data Description
- 156 weeks (3 years, 2022–2024) — enough history for seasonality and trend
- 5 channels: paid search, social, email, display, TV — matching your GitHub description
- Realistic spend patterns: TV is bursty/seasonal, paid search is always-on, email is low-cost high-frequency, display is moderate, social ramps up over time
- Ground truth baked in: I'll build the data-generating process so paid search is genuinely over-saturated, TV has long adstock decay, email has high ROAS but low ceiling, and social is the underinvested channel — which will make the budget optimizer recommendation interesting and defensible
- Confounders included: seasonality (cycling is Q1/Q2 heavy), a trend, and some noise

In [1]:
import numpy as np
import pandas as pd

rng = np.random.default_rng(42)

# ── Time index ────────────────────────────────────────────────────────────────
n = 156  # 3 years of weekly data
dates = pd.date_range("2022-01-03", periods=n, freq="W-MON")
week_of_year = dates.isocalendar().week.to_numpy().astype(float)
year_idx = (dates.year - 2022).to_numpy().astype(float)  # 0, 1, 2

# ── Seasonality (cycling peaks Jan–Apr, secondary Oct) ────────────────────────
seasonality = (
    1.20 * np.sin(2 * np.pi * (week_of_year - 4) / 52)   # primary Jan/Feb peak
  + 0.25 * np.sin(4 * np.pi * (week_of_year - 4) / 52)   # secondary harmonic
)

# ── Long-run trend (modest 8% annual revenue growth) ─────────────────────────
trend = 0.0015 * np.arange(n)

# ── Spend patterns ────────────────────────────────────────────────────────────
# Paid search: always-on, gradually increasing, heavy Q1
paid_search = (
    18_000
    + 4_000 * year_idx
    + 6_000 * np.clip(np.sin(2 * np.pi * (week_of_year - 4) / 52), 0, None)
    + rng.normal(0, 1_500, n)
).clip(8_000)

# Social: low early, ramps significantly by year 3, spiky campaigns
social_base = 8_000 + 5_000 * year_idx
campaign_spikes = rng.choice([0, 1], size=n, p=[0.75, 0.25]) * rng.uniform(5_000, 15_000, n)
social = (social_base + campaign_spikes + rng.normal(0, 1_000, n)).clip(2_000)

# Email: low spend, consistent, slight Q1 push (cost = deployment + creative)
email = (
    2_500
    + 800 * np.clip(np.sin(2 * np.pi * (week_of_year - 6) / 52), 0, None)
    + rng.normal(0, 300, n)
).clip(1_000)

# Display: moderate, always-on retargeting, modest growth
display = (
    9_000
    + 1_500 * year_idx
    + rng.normal(0, 1_200, n)
).clip(4_000)

# TV: bursty — two flights per year (Q1 brand push + Q4 holiday), else zero
tv = np.zeros(n)
for yr in range(3):
    base_week = yr * 52
    # Q1 flight: weeks 2–8
    q1 = np.arange(base_week + 2, min(base_week + 9, n))
    tv[q1] = rng.uniform(60_000, 120_000, len(q1))
    # Q4 flight: weeks 40–47
    q4 = np.arange(base_week + 40, min(base_week + 48, n))
    tv[q4] = rng.uniform(50_000, 100_000, len(q4))

# ── Adstock transformations ───────────────────────────────────────────────────
def adstock(x, decay):
    """Geometric adstock — carryover effect of advertising."""
    out = np.zeros_like(x, dtype=float)
    out[0] = x[0]
    for t in range(1, len(x)):
        out[t] = x[t] + decay * out[t - 1]
    return out

paid_search_adstocked = adstock(paid_search, decay=0.3)   # short memory
social_adstocked       = adstock(social,      decay=0.4)   # moderate
email_adstocked        = adstock(email,        decay=0.2)   # very short
display_adstocked      = adstock(display,      decay=0.35)  # moderate
tv_adstocked           = adstock(tv,           decay=0.7)   # long brand halo

# ── Hill (saturation) transformation ─────────────────────────────────────────
def hill(x, k, n_hill):
    """Hill function: diminishing returns. k = half-saturation point."""
    return x**n_hill / (k**n_hill + x**n_hill)

# Normalize each channel to [0, spend_scale] before Hill so k is interpretable
def hill_scaled(x, k_pct=0.6, n_hill=2.0):
    """k as percentile of observed spend, so saturation is data-relative."""
    k = np.percentile(x[x > 0], k_pct * 100)
    return hill(x, k, n_hill)

ps_sat   = hill_scaled(paid_search_adstocked, k_pct=0.45, n_hill=3.0)  # heavy saturation
soc_sat  = hill_scaled(social_adstocked,       k_pct=0.75, n_hill=1.8)  # under-saturated
em_sat   = hill_scaled(email_adstocked,         k_pct=0.55, n_hill=2.2)
disp_sat = hill_scaled(display_adstocked,       k_pct=0.60, n_hill=2.0)
tv_sat   = hill_scaled(tv_adstocked,            k_pct=0.65, n_hill=1.5)  # linear-ish (bursty)

# ── Revenue DGP ───────────────────────────────────────────────────────────────
# Coefficients represent max incremental revenue contribution (at full saturation)
# Paid search: high absolute but heavily saturated → low marginal ROAS
# Social: moderate spend but under-saturated → high marginal ROAS
# Email: tiny spend, great marginal ROAS, low ceiling
# TV: big brand halo, hard to attribute short-term
BASELINE = 280_000   # weekly baseline revenue (zero-marketing floor)

revenue = (
    BASELINE
    + 80_000  * ps_sat       # paid search — large coef but flattened
    + 70_000  * soc_sat      # social — meaningful, still on steep part of curve
    + 40_000  * em_sat       # email — punches above its weight
    + 35_000  * disp_sat     # display — supporting role
    + 90_000  * tv_sat       # TV — biggest max effect, long decay
    + 60_000  * seasonality  # seasonal swing
    + 15_000  * trend        # growth trend
    + rng.normal(0, 12_000, n)  # noise
)

# ── Assemble dataframe ────────────────────────────────────────────────────────
df = pd.DataFrame({
    "week":         dates,
    "revenue":      revenue.round(2),
    "spend_paid_search": paid_search.round(2),
    "spend_social":      social.round(2),
    "spend_email":       email.round(2),
    "spend_display":     display.round(2),
    "spend_tv":          tv.round(2),
})

df.to_csv("arclight_mmm_data.csv", index=False)
print(df.describe().round(0).to_string())
print(f"\nTotal rows: {len(df)}")
print(f"Date range: {df.week.min().date()} → {df.week.max().date()}")
print(f"\nMean weekly revenue:     ${df.revenue.mean():,.0f}")
print(f"Mean weekly total spend: ${(df[['spend_paid_search','spend_social','spend_email','spend_display','spend_tv']].sum(axis=1).mean()):,.0f}")

                      week   revenue  spend_paid_search  spend_social  spend_email  spend_display  spend_tv
count                  156     156.0              156.0         156.0        156.0          156.0     156.0
mean   2023-06-29 12:00:00  415868.0            23819.0       15678.0       2731.0        10428.0   23666.0
min    2022-01-03 00:00:00  277597.0            15797.0        5927.0       1804.0         6856.0       0.0
25%    2022-10-01 06:00:00  366490.0            20607.0       11853.0       2402.0         9219.0       0.0
50%    2023-06-29 12:00:00  405294.0            23664.0       15312.0       2691.0        10492.0       0.0
75%    2024-03-26 18:00:00  463186.0            26642.0       18740.0       3049.0        11599.0   60899.0
max    2024-12-23 00:00:00  570690.0            34228.0       32077.0       3992.0        14472.0  116401.0
std                    NaN   62926.0             4170.0        6135.0        457.0         1645.0   38402.0

Total rows: 156
Date range: